In [1]:
from pyspark.sql import SparkSession
from pyspark import SparkFiles

# Inicializar sesión
spark = SparkSession.builder.appName("RegresionStartups").getOrCreate()

# Cargar el dataset de 50 Startups
url_startups = "https://raw.githubusercontent.com/gakudo-ai/open-datasets/refs/heads/main/50_Startups.csv"
spark.sparkContext.addFile(url_startups)

df_startups = spark.read.csv(SparkFiles.get("50_Startups.csv"), header=True, inferSchema=True)

# Limpiamos nombres de columnas para evitar problemas con los espacios
df_startups = df_startups.withColumnRenamed("R&D Spend", "RD_Spend") \
                         .withColumnRenamed("Marketing Spend", "Marketing_Spend")

df_startups.show(5)

+---------+--------------+---------------+----------+---------+
| RD_Spend|Administration|Marketing_Spend|     State|   Profit|
+---------+--------------+---------------+----------+---------+
| 165349.2|      136897.8|       471784.1|  New York|192261.83|
| 162597.7|     151377.59|      443898.53|California|191792.06|
|153441.51|     101145.55|      407934.54|   Florida|191050.39|
|144372.41|     118671.85|      383199.62|  New York|182901.99|
|142107.34|      91391.77|      366168.42|   Florida|166187.94|
+---------+--------------+---------------+----------+---------+
only showing top 5 rows


In [2]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Preparar la variable independiente (Feature) en formato Vector
assembler_simple = VectorAssembler(inputCols=["RD_Spend"], outputCol="features_simple")
df_simple = assembler_simple.transform(df_startups)

# 2. Dividir los datos en conjunto de Entrenamiento (80%) y Prueba (20%)
train_simple, test_simple = df_simple.randomSplit([0.8, 0.2], seed=42)

# 3. Inicializar y Entrenar el modelo de Regresión Lineal
lr_simple = LinearRegression(featuresCol="features_simple", labelCol="Profit")
modelo_simple = lr_simple.fit(train_simple)

# 4. Realizar predicciones sobre el conjunto de prueba
predicciones_simple = modelo_simple.transform(test_simple)
predicciones_simple.select("RD_Spend", "Profit", "prediction").show(5)

# 5. Evaluar el modelo (R2)
evaluator = RegressionEvaluator(labelCol="Profit", predictionCol="prediction", metricName="r2")
r2_simple = evaluator.evaluate(predicciones_simple)
print(f"R Cuadrado (Regresión Simple): {r2_simple:.4f}")

+--------+--------+------------------+
|RD_Spend|  Profit|        prediction|
+--------+--------+------------------+
|  542.05|35673.41| 50427.33354978463|
|20229.59|81229.06| 66772.25497517604|
|23640.93|71498.49| 69604.40588155107|
|44069.95|89949.14| 86564.91627024104|
|63408.86|97427.84|102620.39930177855|
+--------+--------+------------------+
only showing top 5 rows
R Cuadrado (Regresión Simple): 0.9646


La métrica R2 (coeficiente de determinación) nos indica qué porcentaje de la variación en la variable a predecir (Profit) es explicada por las variables predictoras.

In [5]:
# 1. Preparar múltiples variables independientes en un solo Vector
assembler_multiple = VectorAssembler(
    inputCols=["RD_Spend", "Administration", "Marketing_Spend"],
    outputCol="features_multiple"
)
df_multiple = assembler_multiple.transform(df_startups)

# 2. Dividir en Entrenamiento (80%) y Prueba (20%)
train_multiple, test_multiple = df_multiple.randomSplit([0.8, 0.2], seed=42)

# 3. Inicializar y Entrenar el modelo
lr_multiple = LinearRegression(featuresCol="features_multiple", labelCol="Profit")
modelo_multiple = lr_multiple.fit(train_multiple)

# 4. Predicciones
predicciones_multiple = modelo_multiple.transform(test_multiple)
predicciones_multiple.select("features_multiple", "Profit", "prediction").show(5)

# 5. Evaluar y comparar con el modelo simple
r2_multiple = evaluator.evaluate(predicciones_multiple)
print(f"R Cuadrado (Regresión Múltiple): {r2_multiple:.4f}")

# Mostrar los coeficientes asignados a cada variable
print(f"Coeficientes (I+D, Admin, Marketing): {modelo_multiple.coefficients}")
print(f"Intercept (peso independiente / términos de sesgo): {modelo_multiple.intercept}")

+--------------------+--------+------------------+
|   features_multiple|  Profit|        prediction|
+--------------------+--------+------------------+
|[542.05,51743.15,...|35673.41|51911.856758620626|
|[20229.59,65947.9...|81229.06| 70023.91011859625|
|[23640.93,96189.6...|71498.49| 71080.02550448562|
|[44069.95,51283.1...|89949.14| 90046.08625930431|
|[63408.86,129219....|97427.84|100493.56397557817|
+--------------------+--------+------------------+
only showing top 5 rows
R Cuadrado (Regresión Múltiple): 0.9715
Coeficientes (I+D, Admin, Marketing): [0.8090774172068323,-0.03821206376451953,0.014714607903703374]
Intercept (peso independiente / términos de sesgo): 53450.50889180076
